### Maps

**Note**: This is about the map data strucutre, not to be confused with the `map` operation in `map`, `fold`, `filter` operations for lists and other datastructures.

In [6]:
(* an interface for maps *)
module type Map = sig

  (** [('k, 'v) t] is the type of maps that bind keys of type
      ['k] to values of type ['v]. *)
  type ('k, 'v) t

  (** [insert k v m] is the same map as [m], but with an additional
      binding from [k] to [v].  If [k] was already bound in [m],
      that binding is replaced by the binding to [v] in the new map. *)
  val insert : 'k -> 'v -> ('k, 'v) t -> ('k, 'v) t

  (** [find k m] is [Some v] if [k] is bound to [v] in [m],
      and [None] if not. *)
  val find : 'k -> ('k, 'v) t -> 'v option

  (** [remove k m] is the same map as [m], but without any binding of [k].
      If [k] was not bound in [m], then the map is unchanged. *)
  val remove : 'k -> ('k, 'v) t -> ('k, 'v) t

  (** [empty] is the empty map. *)
  val empty : ('k, 'v) t

  (** [of_list lst] is a map containing the same bindings as
      association list [lst].
      Requires: [lst] does not contain any duplicate keys. *)
      (* This converts a list to a map. *)
  val of_list : ('k * 'v) list -> ('k, 'v) t

  (** [bindings m] is an association list containing the same
      bindings as [m]. There are no duplicates in the list. *)
      (* This converts a map to a list. *)
  val bindings : ('k, 'v) t -> ('k * 'v) list

end

module type Map =
  sig
    type ('k, 'v) t
    val insert : 'k -> 'v -> ('k, 'v) t -> ('k, 'v) t
    val find : 'k -> ('k, 'v) t -> 'v option
    val remove : 'k -> ('k, 'v) t -> ('k, 'v) t
    val empty : ('k, 'v) t
    val of_list : ('k * 'v) list -> ('k, 'v) t
    val bindings : ('k, 'v) t -> ('k * 'v) list
  end


In [7]:
(* The simplest implementation of a map in OCaml is as an association list. *)
module ListMap : Map = struct
  (** AF: [[(k1, v1); (k2, v2); ...; (kn, vn)]] is the map {k1 : v1, k2 : v2,
      ..., kn : vn}. If a key appears more than once in the list, then in the
      map it is bound to the left-most occurrence in the list. For example,
      [[(k, v1); (k, v2)]] represents {k : v1}. The empty list represents
      the empty map.
      RI: none. *)
  type ('k, 'v) t = ('k * 'v) list

  (** Efficiency: O(1). *)
  let insert k v m = (k, v) :: m

  (** Efficiency: O(n). *)
  let find = List.assoc_opt (* https://v2.ocaml.org/api/List.html *) 
    (* Optional Association *) 
    (* val assoc_opt : 'a -> ('a * 'b) list -> 'b option *)

  (** Efficiency: O(n). *)
  let remove k lst = List.filter (fun (k', _) -> k <> k') lst

  (** Efficiency: O(1). *)
  let empty = []

  (** Efficiency: O(1). *)
  let of_list lst = lst

  (** [keys m] is a list of the keys in [m], without
      any duplicates.
      Efficiency: O(n log n). *)
  let keys m = m |> List.map fst |> List.sort_uniq Stdlib.compare

  (** [binding m k] is [(k, v)], where [v] is the value that [k]
       binds in [m].
       Requires: [k] is a key in [m].
       Efficiency: O(n). *)
  let binding m k = (k, List.assoc k m)

  (** Efficiency: O(n log n) + O(n) * O(n), which is O(n^2). *)
  let bindings m = List.map (binding m) (keys m)
end

module ListMap : Map


In [8]:
ListMap.find 1 (ListMap.of_list [(1, 2); (3, 4)]) (* Some 2 *);;
let lm = ListMap.insert 5 6 (ListMap.of_list [(1, 2); (3, 4)]);;
ListMap.find 5 lm;; (* Some 6 *)
ListMap.find 1 lm;; (* Some 2 *)
ListMap.find 7 lm;; 

- : int option = Some 2


val lm : (int, int) ListMap.t = <abstr>


- : int option = Some 6


- : int option = Some 2


- : int option = None


### Hash Tables

Hash tables are hashed association tables, with in-place modification.

https://v2.ocaml.org/api/Hashtbl.html

In [29]:
let my_hash = Hashtbl.create 1000;;

val my_hash : ('_weak3, '_weak4) Hashtbl.t = <abstr>


In [30]:
my_hash;;

- : ('_weak3, '_weak4) Hashtbl.t = <abstr>


In [31]:
  Hashtbl.add my_hash "h" "hello";
  Hashtbl.add my_hash "h" "hi";
  Hashtbl.add my_hash "h" "hug";
  Hashtbl.add my_hash "h" "hard";
  Hashtbl.add my_hash "w" "wimp";
  Hashtbl.add my_hash "w" "world";
  Hashtbl.add my_hash "w" "wine";;

- : unit = ()


In [32]:
  Hashtbl.add my_hash "w" [1; 2];;

error: compile_error

In [33]:
Hashtbl.find my_hash "h";;

- : string = "hard"


In [34]:
Hashtbl.find_all my_hash "h";;

- : string list = ["hard"; "hug"; "hi"; "hello"]


In [35]:
Hashtbl.remove my_hash "h";;

- : unit = ()


In [36]:
Hashtbl.find my_hash "h";;

- : string = "hug"


In [37]:
Hashtbl.find_all my_hash "h";;

- : string list = ["hug"; "hi"; "hello"]


In [38]:
Hashtbl.replace my_hash "t" "try";

- : unit = ()


In [39]:
Hashtbl.replace my_hash "t" "test";

- : unit = ()


In [40]:
Hashtbl.find_all my_hash "t";;

- : string list = ["test"]


In [41]:
Hashtbl.remove my_hash "t";

- : unit = ()


In [42]:
Hashtbl.find my_hash "t";;

error: runtime_error

In [43]:
Hashtbl.find_opt my_hash "t";;

- : string option = None


In [44]:
Hashtbl.find_opt my_hash "h";;

- : string option = Some "hug"


In [45]:
(* Here's the C++/Java way of dealing with exceptions *)
try
  Hashtbl.find my_hash "t"
with
  | Not_found -> print_endline "NOT FOUND."; "" 

NOT FOUND.


- : string = ""


In [46]:
try
  Hashtbl.find my_hash "h"
with
  | Not_found -> print_endline "NOT FOUND."; "" 

- : string = "hug"


In [47]:
(* This equivalent method is cleaner and more programmer-friendly. *)
match Hashtbl.find my_hash "t" with
| result -> result
| exception Not_found -> ""

- : string = ""


In [48]:
match Hashtbl.find my_hash "h" with
| result -> result
| exception Not_found -> ""

- : string = "hug"


In [49]:
Hashtbl.mem my_hash "h";;

- : bool = true


In [50]:
Hashtbl.mem my_hash "t";;

- : bool = false


In [51]:
Hashtbl.stats my_hash

- : Hashtbl.statistics =
{Hashtbl.num_bindings = 6; num_buckets = 1024; max_bucket_length = 3;
 bucket_histogram = [|1022; 0; 0; 2|]}


In [52]:
Hashtbl.hash [1; 2; 3; 4; 5;];;
Hashtbl.hash [1; 2; 3; 4; 6;];;
Hashtbl.hash [1; 2; 3; 4; 5; 6; ];;
Hashtbl.hash [1; 2; 3; 4; 5; 6; 7; 8; 9];;
Hashtbl.hash [1; 2; 3; 4; 5; 6; 7; 8; 9; 10];;

(* Hashtbl.hash x associates a nonnegative integer to any value of any type. It is guaranteed that if x = y or Stdlib.compare x y = 0, then hash x = hash y. Moreover, hash always terminates, even on cyclic structures. *)

- : int = 468619635


- : int = 103498579


- : int = 418354221


- : int = 635296333


- : int = 822221246


In [53]:
Hashtbl.hash 3

- : int = 152507349


In [54]:
Hashtbl.hash "Hello"

- : int = 200495445


In [55]:
Hashtbl.hash 100.22

- : int = 772367056


### Hashtbl directly implements all features necessary for Module type Map.

### Red Black Tree

It’s easy to use a BST or a Red Black Tree to implement either a map or a set ADT.

In [56]:
type color = Red | Black
type 'a rbtree = Leaf | Node of color * 'a * 'a rbtree * 'a rbtree

type color = Red | Black


type 'a rbtree = Leaf | Node of color * 'a * 'a rbtree * 'a rbtree


In [ ]:
let rec mem x = function
  | Leaf -> false
  | Node (_, y, l, r) ->
    if x < y then mem x l
    else if x > y then mem x r
    else true

val mem : 'a -> 'a rbtree -> bool = <fun>


In [ ]:
let balance = function
  | Black, z, Node (Red, y, Node (Red, x, a, b), c), d
  | Black, z, Node (Red, x, a, Node (Red, y, b, c)), d
  | Black, x, a, Node (Red, z, Node (Red, y, b, c), d)
  | Black, x, a, Node (Red, y, b, Node (Red, z, c, d)) ->
    Node (Red, y, Node (Black, x, a, b), Node (Black, z, c, d))
  | a, b, c, d -> Node (a, b, c, d)

val balance : color * 'a * 'a rbtree * 'a rbtree -> 'a rbtree = <fun>


In [ ]:
let insert x s =
  let rec ins = function
    | Leaf -> Node (Red, x, Leaf, Leaf)
    | Node (color, y, a, b) as s ->
      if x < y then balance (color, y, ins a, b)
      else if x > y then balance (color, y, a, ins b)
      else s
  in
  match ins s with
  | Node (_, y, a, b) -> Node (Black, y, a, b)
  | Leaf -> (* guaranteed to be nonempty *)
    failwith "RBT insert failed with ins returning leaf"

val insert : 'a -> 'a rbtree -> 'a rbtree = <fun>


In [ ]:
let t = insert 10 Leaf;;


val t : int rbtree = Node (Black, 10, Leaf, Leaf)


In [ ]:
let t = insert 20 t;;


val t : int rbtree = Node (Black, 10, Leaf, Node (Red, 20, Leaf, Leaf))


In [ ]:
let t = insert 15 t;;

val t : int rbtree =
  Node (Black, 15, Node (Black, 10, Leaf, Leaf),
   Node (Black, 20, Leaf, Leaf))


In [ ]:
let t = Node (Black, 15, 
            Node (Black, 10, Leaf, Leaf),
            Node (Black, 20, Leaf, Leaf)
            )

val t : int rbtree =
  Node (Black, 15, Node (Black, 10, Leaf, Leaf),
   Node (Black, 20, Leaf, Leaf))


In [ ]:
let t = insert 18 t

val t : int rbtree =
  Node (Black, 15, Node (Black, 10, Leaf, Leaf),
   Node (Black, 20, Node (Red, 18, Leaf, Leaf), Leaf))


In [ ]:
Hashtbl.hash t

- : int = 388305121


In [ ]:
(* This is trivial but helps provide some convience and readability. *)

let empty = Leaf

val empty : 'a rbtree = Leaf


In [ ]:
let t = insert 10 empty;;

val t : int rbtree = Node (Black, 10, Leaf, Leaf)


### Exercise: How to write the deletion function?

### Helpful Links for RBTree:
RBT Visualization: https://www.cs.usfca.edu/~galles/visualization/RedBlack.html

An alternative RBT implementation: https://github.com/CompScienceClub/ocaml-red-black-trees/blob/master/src/red_black_tree.ml

### Memoization

https://cs3110.github.io/textbook/chapters/ds/memoization.html

In [57]:
let rec fib n = if n < 2 then 1 else fib (n - 1) + fib (n - 2)


val fib : int -> int = <fun>


In [58]:
fib 40

(* It takes several seconds to compute every time this function was called with the parameter 40, 
   even though the result will always be the same. *)

- : int = 165580141


In [59]:
(* Here is a version with memoization. *)

let fibm n =
  let memo : int option array = Array.make (n + 1) None in
  let rec f_mem n =
    match memo.(n) with
    | Some result -> (* computed already *) result
    | None ->
        let result =
          if n < 2 then 1 else f_mem (n - 1) + f_mem (n - 2)
        in
        (* record in table *)
        memo.(n) <- Some result;
        result
  in
  f_mem n

  (* https://v2.ocaml.org/api/Array.html *)

val fibm : int -> int = <fun>


In [60]:
fibm 40

- : int = 165580141


In [61]:
fibm 50

- : int = 20365011074


In [35]:
(* In comparison, here's how long it will take for the non-memoization version. 
   Between 5 to 10 minutes. Fast enough to finish within the lecture session. *)

fib 50

- : int = 20365011074


In [ ]:
(* We will have enough time to discuss this section in detail while this function runs. *)
fib 200

In [62]:
fibm 100

(* F100 = 3.5422484817926E+20 *)

- : int = 1298777728820984005


In [63]:
fibm 200

(* F200 = 173402521172797813159685037284371942044301 + 107168651819712326877926895128666735145224 *)
(* F200 = 2.8057117299251E+41 *)

- : int = 3721511182311577122


### In-class challenge: How to re-write fibm using Big Numbers? (more on this later on this page.)
### In-class challenge: How to re-write fibm using Hashtbl?
...

Ch8.5.2. Memoization Using Higher-order Functions (such as Hashtbl)

### In-class challenge 2: How to make this method generic for other functions, not just hard-coding fib inside?

In [1]:
let memo_rec f =
  let h = Hashtbl.create 16 in
  let rec g x =
    try 
        Hashtbl.find h x
    with Not_found ->
      let y = f g x in
          Hashtbl.add h x y;
          y
  in
  g

val memo_rec : (('a -> 'b) -> 'a -> 'b) -> 'a -> 'b = <fun>


In [2]:
let fib_memo =
  let fib self n =
    if n < 2 then 1 else self (n - 1) + self (n - 2)
  in
  memo_rec fib

val fib_memo : int -> int = <fun>


In [3]:
fib_memo 40

- : int = 165580141


In [4]:
fib_memo 50

- : int = 20365011074


In [5]:
fib_memo 100

- : int = 1298777728820984005


In [6]:
fib_memo 200

- : int = 3721511182311577122


### Challenge 3: How to make fib work for very big numbers?

In [6]:
(* 2023: no longer need this *) 

#use "topfind"

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()


In [ ]:
(* 2023: no longer need this *)

#use "/home/codespace/.opam/default/lib/toplevel/topfind"  
(* It seems codespaces need this path explicityly. *) 

Cannot find file /home/codespace/.opam/default/lib/toplevel/topfind.


In [8]:
#require "num";;

In [6]:
#list;;

astring             (version: 0.8.5)
astring.top         (version: 0.8.5)
base                (version: v0.16.3)
base.base_internalhash_types (version: v0.16.3)
base.caml           (version: v0.16.3)
base.md5            (version: v0.16.3)
base.shadow_stdlib  (version: v0.16.3)
base64              (version: 3.5.1)
base64.rfc2045      (version: 3.5.1)
base_bigstring      (version: v0.16.0)
base_quickcheck     (version: v0.16.0)
base_quickcheck.ppx_quickcheck (version: v0.16.0)
base_quickcheck.ppx_quickcheck.expander (version: v0.16.0)
base_quickcheck.ppx_quickcheck.runtime (version: v0.16.0)
bigarray            (version: [distributed with Ocaml])
bignum              (version: v0.16.0)
bignum.bigint       (version: v0.16.0)
bin_prot            (version: v0.16.0)
bin_prot.shape      (version: v0.16.0)
bin_prot.xen        (version: v0.16.0)
bytes               (version: [distributed with OCaml 4.02 or above])
camlp-streams       (version: n/a)
chrome-trace        (version: 3.10.0)
cmdliner 

In [9]:
open Num;;

In [10]:

let rec fib n = if n < 2 then num_of_int 1 else fib (n - 1) +/ fib (n - 2)


val fib : int -> Num.num = <fun>


In [9]:
fib 40

- : Num.num = <num 165580141>


In [10]:
let memo_rec f =
  let h = Hashtbl.create 16 in
  let rec g x =
    try Hashtbl.find h x
    with Not_found ->
      let y = f g x in
      Hashtbl.add h x y;
      y
  in
  g

val memo_rec : (('a -> 'b) -> 'a -> 'b) -> 'a -> 'b = <fun>


In [11]:
let fib_memo =
  let rec fib self n =
    if n < 2 then num_of_int 1 else self (n - 1) +/ self (n - 2)
  in
  memo_rec fib

val fib_memo : int -> Num.num = <fun>


In [12]:
fib_memo 40

- : Num.num = <num 165580141>


In [13]:
fib_memo 100

- : Num.num = <num 573147844013817084101>


In [14]:
fib_memo 200

- : Num.num = <num 453973694165307953197296969697410619233826>


In [15]:
fib_memo 500

- : Num.num =
<num 225591516161936330872512695036072072046011324913758190588638866418474627738686883405015987052796968498626>


In [16]:
fib_memo 2000

- : Num.num =
<num 6835702259575806647045396549170580107055408029365524565407553367798082454408054014954534318953113802726603726769523447478238192192714526677939943338306101405105414819705664090901813637296453767095528104868264704914433529355579148731044685634135487735897954629842516947101494253575869699893400976539545740214819819151952085089538422954565146720383752121972115725761141759114990448978941370030912401573418221496592822626>


In [17]:
fib_memo 20000

- : Num.num =
<num 409550667084212509197492036759335074992840743265840116213317525939589358561297513435763135796790214774111472601629822174152083112130913819966482586418785450132812012214804353617615506745431997339175336752451614146223272543518539296853983476607804818554327616296404625185956586805573767964118906858499124928263915208320344881583310416638326135924677565665469946282685882787510680479312550566478984709771528600548025585011856296386121368499368874426983851837687004668449037159360312267783617951142133528096385526989046736769343367788472198018698638510799062589847769316871278970705895373752015116223385578103374294311071644062913041730400722835242427376264512018286784789317039343555108736376931281808388418367079925026061444328905998020987515950017577921862556131588409835947354688009592801555031122157674748999010552313630592664178185579283310790692824533581022613466263295047412962592591974538519151568310968046557220822319166225584830482005298149827209763900369246743511092410269

### Sum with big number

In [26]:
open Num

let sum lst =
  let rec sum_tail_recursive acc lst =
    match lst with
    | [] -> acc
    | h :: t -> sum_tail_recursive (add_num acc (num_of_int h)) t
  in
  sum_tail_recursive (num_of_int 0) lst
;;

let ( -- ) i j = 
  let rec f i j acc = 
    if i > j then 
      acc 
    else 
      f (i+1)  (j) ( i :: acc);
  in
  f i j []
;;


val sum : int list -> Num.num = <fun>


val ( -- ) : int -> int -> int list = <fun>


In [30]:
0--4;;
sum (0--100000000)

- : int list = [4; 3; 2; 1; 0]


- : Num.num = <num 5000000050000000>


In [16]:
0--100;;
sum (0--100);;

- : int list =
[0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0;
 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1;
 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2;
 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3; 0; 1; 2; 3;
 0]


- : Num.num = <num 150>
